# Combined Sealed-Test Evaluation — GraphSAGE: DEFAULT vs REGIME-ALIGNED TUNED

Evaluates nine configurations once each on the sealed test set:

- `{PSO,HHO,MI,ALL}_DEFAULT` — each feature subset under the frozen Track-1 default
  hyperparameters (the sealed-test anchor). Isolates the feature-selection effect.
- `{PSO,HHO,MI,ALL}_TUNED` — each subset under its regime-aligned enhanced Optuna
  hyperparameters (from `enhanced-optuna-regime-aligned`). The default-vs-tuned
  delta per subset is a clean test of whether tuning helps under the deployment regime.
- `BASELINE` — all 48 features under defaults; the identity twin of `ALL_DEFAULT`
  (must match to <1e-9; protocol identity check in Cell 8).


In [ ]:
# CELL 1 — INSTALLATION
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args, '-q'], check=False)

pip('torch_geometric', 'numpy>=2')
pip('imbalanced-learn', 'numpy>=2')

print("Installation complete. Restart the kernel, then run Cell 2 onward.")

In [1]:
# CELL 2 — IMPORTS, SEED, CONFIG
import os, gc, json, random, warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.nn import Linear

import torch_geometric
from torch_geometric.data  import Data
from torch_geometric.nn    import SAGEConv
from torch_geometric.utils import coalesce
from gensim.models         import Word2Vec

from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
    cohen_kappa_score, confusion_matrix, fowlkes_mallows_score,
    normalized_mutual_info_score,
)
from imblearn.over_sampling import SMOTE

print(f"PyTorch           : {torch.__version__}")
print(f"PyTorch Geometric : {torch_geometric.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

GLOBAL_SEED = 42
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(GLOBAL_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device            : {DEVICE}")

# ── Paths — update these to match your Kaggle dataset layout ─────
PATHS = {
    'DATA'          : ('/kaggle/input/datasets/monamehrun/'
                       'pcos-cleaned-dataset/pcos_cleaned.csv'),
    'ENHANCED_HP'   : ('/kaggle/input/datasets/galibbhai/'
                       'enhanced-optuna/best_hyperparams_graphsage_enhanced.json'),
    'SELECTED_FEAT' : ('/kaggle/input/datasets/galibbhai/'
                       'selected-features/selected_features.json'),
}
OUTPUT_DIR = '/kaggle/working/'

# ── Fixed config shared by all runs ──────────────────────────────
FIXED = {
    'TARGET_COL'   : 'PCOS',
    'TEST_SIZE'    : 0.20,
    'SEED'         : GLOBAL_SEED,
    'N2V_WALK_LEN' : 20,
    'N2V_CONTEXT'  : 10,
    'N2V_WALKS'    : 10,
    'N2V_EPOCHS'   : 50,
    'N2V_LR'       : 0.01,
    'FINAL_EPOCHS' : 150,     # fixed — no early stopping (matches Track-1 protocol)
}

print("Configuration loaded.")
print(f"Training epochs : {FIXED['FINAL_EPOCHS']} (fixed, no early stopping)")

PyTorch           : 2.10.0+cu128
PyTorch Geometric : 2.8.0
CUDA available    : True
Device            : cuda
Configuration loaded.
Training epochs : 150 (fixed, no early stopping)


In [2]:
# CELL 3 — GRAPH, NODE2VEC AND SMOTE UTILITIES
# Verbatim from the validated final-evaluation / CV notebooks. Do not modify.

def build_knn_graph(features_scaled: np.ndarray, k: int = 10):
    n   = len(features_scaled)
    sim = cosine_similarity(features_scaled)
    np.fill_diagonal(sim, -2.0)
    src, dst, wts = [], [], []
    for i in range(n):
        top_k = np.argpartition(sim[i], -k)[-k:]
        for j in top_k:
            w = float(max(0.0, sim[i][j]))
            src += [i, j];  dst += [j, i];  wts += [w, w]
    ei = torch.tensor([src, dst], dtype=torch.long)
    ew = torch.tensor(wts,        dtype=torch.float)
    ei, ew = coalesce(ei, ew, num_nodes=n, reduce='max')
    return ei, ew


def add_synthetic_nodes(ei, ew, X_real_sc, X_syn_sc, k=10):
    n_real, n_syn = len(X_real_sc), len(X_syn_sc)
    if n_syn == 0:
        return ei, ew
    sim = cosine_similarity(X_syn_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_syn):
        s = n_real + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [s, j];  dst += [j, s];  wts += [w, w]
    new_ei = torch.tensor([src, dst], dtype=torch.long)
    new_ew = torch.tensor(wts,        dtype=torch.float)
    aug_ei = torch.cat([ei, new_ei], dim=1)
    aug_ew = torch.cat([ew, new_ew])
    aug_ei, aug_ew = coalesce(aug_ei, aug_ew,
                               num_nodes=n_real + n_syn, reduce='max')
    return aug_ei, aug_ew


def add_test_nodes(ei_aug, ew_aug, X_real_sc, X_test_sc,
                    k=10, n_train_aug=None):
    """Identical to add_val_nodes — renamed for clarity in the test context."""
    n_real = len(X_real_sc)
    if n_train_aug is None:
        n_train_aug = n_real
    sim = cosine_similarity(X_test_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(len(X_test_sc)):
        v = n_train_aug + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [v, j];  dst += [j, v];  wts += [w, w]
    new_ei  = torch.tensor([src, dst], dtype=torch.long)
    new_ew  = torch.tensor(wts,        dtype=torch.float)
    comb_ei = torch.cat([ei_aug, new_ei],  dim=1)
    comb_ew = torch.cat([ew_aug, new_ew])
    return comb_ei, comb_ew


def _random_walks(edge_index, num_nodes, walk_length, walks_per_node, seed=42):
    import random as _r
    _r.seed(seed)
    adj = [[] for _ in range(num_nodes)]
    ei  = edge_index.cpu().numpy()
    for s, d in zip(ei[0], ei[1]):
        adj[int(s)].append(int(d))
    walks = []
    nodes = list(range(num_nodes))
    for _ in range(walks_per_node):
        _r.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                curr = walk[-1]
                nbrs = adj[curr]
                if nbrs:
                    walk.append(_r.choice(nbrs))
                else:
                    break
            walks.append([str(n) for n in walk])
    return walks


def train_node2vec(edge_index, num_nodes: int, cfg: dict):
    walks = _random_walks(edge_index, num_nodes,
                          walk_length    = cfg['N2V_WALK_LEN'],
                          walks_per_node = cfg['N2V_WALKS'],
                          seed           = cfg['SEED'])
    w2v = Word2Vec(sentences   = walks,
                   vector_size = cfg['N2V_DIM'],
                   window      = cfg['N2V_CONTEXT'],
                   min_count   = 0,
                   sg          = 1,
                   workers     = 1,
                   seed        = cfg['SEED'],
                   epochs      = cfg['N2V_EPOCHS'])
    emb = np.zeros((num_nodes, cfg['N2V_DIM']), dtype=np.float32)
    for idx in range(num_nodes):
        key = str(idx)
        if key in w2v.wv:
            emb[idx] = w2v.wv[key]
    return emb


def inductive_n2v(X_new_sc, X_train_sc, n2v_train, k=10):
    sim = cosine_similarity(X_new_sc, X_train_sc)
    out = np.zeros((len(X_new_sc), n2v_train.shape[1]), dtype=np.float32)
    for i in range(len(X_new_sc)):
        top_k = np.argpartition(sim[i], -k)[-k:]
        w     = np.maximum(sim[i][top_k], 0.0)
        wsum  = w.sum()
        w     = w / wsum if wsum > 1e-9 else np.ones(k) / k
        out[i] = (n2v_train[top_k] * w[:, None]).sum(axis=0)
    return out


def apply_smote(X, y, seed=42):
    smote        = SMOTE(random_state=seed, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X, y)
    n_syn = len(X_res) - len(X)
    print(f"  SMOTE: +{n_syn} synthetic PCOS+ "
          f"({len(X)} -> {len(X_res)} total)")
    return X_res, y_res


print("Graph / Node2Vec / SMOTE utilities loaded.")

Graph / Node2Vec / SMOTE utilities loaded.


In [3]:
# CELL 4 — MODEL DEFINITION + TRAINING / EVALUATION HELPERS

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1   = SAGEConv(in_ch, hidden_ch)
        self.c2   = SAGEConv(hidden_ch, hidden_ch)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


def train_one_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index, data.edge_weight)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    if torch.isnan(loss):
        return float('nan')
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_model(model, data, mask):
    model.eval()
    out   = model(data.x, data.edge_index, data.edge_weight)
    probs = torch.softmax(out[mask], dim=1)[:, 1].cpu().numpy()
    preds = out[mask].argmax(dim=1).cpu().numpy()
    true  = data.y[mask].cpu().numpy()
    return preds, probs, true


def compute_metrics(true, preds, probs):
    cm = confusion_matrix(true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    try:    auroc = roc_auc_score(true, probs)
    except: auroc = 0.0
    return {
        'accuracy'    : accuracy_score(true, preds),
        'precision'   : precision_score(true, preds, zero_division=0),
        'recall'      : recall_score(true, preds, zero_division=0),
        'f1'          : f1_score(true, preds, zero_division=0),
        'mcc'         : matthews_corrcoef(true, preds),
        'auroc'       : auroc,
        'auprc'       : average_precision_score(true, probs),
        'sensitivity' : sens,
        'specificity' : spec,
        'kappa'       : cohen_kappa_score(true, preds),
        'fmi'         : fowlkes_mallows_score(true, preds),
        'nmi'         : normalized_mutual_info_score(true, preds),
    }, (int(tn), int(fp), int(fn), int(tp))


print("GraphSAGE + train / eval / metrics loaded.")

GraphSAGE + train / eval / metrics loaded.


In [4]:
# CELL 5 — DATA LOADING, SPLIT, NINE-CONFIG ASSEMBLY (default vs tuned + BASELINE)
df = pd.read_csv(PATHS['DATA'])
print(f"Loaded  : {df.shape[0]} rows x {df.shape[1]} columns")

y_full        = df[FIXED['TARGET_COL']].values.astype(np.int64)
X_full        = df.drop(columns=[FIXED['TARGET_COL']]).values.astype(np.float32)
feature_names = df.drop(columns=[FIXED['TARGET_COL']]).columns.tolist()
name_to_idx   = {name: i for i, name in enumerate(feature_names)}
print(f"Features: {len(feature_names)}")
print(f"PCOS=0  : {(y_full==0).sum()}   PCOS=1  : {(y_full==1).sum()}")

# Outer split — IDENTICAL to the tuning notebook and Track-1 (test_size=0.20, seed=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size    = FIXED['TEST_SIZE'],
    stratify     = y_full,
    random_state = FIXED['SEED'],
)
print(f"\nTraining pool  : {len(X_train)} patients  "
      f"(PCOS=0: {(y_train==0).sum()}  PCOS=1: {(y_train==1).sum()})")
print(f"Sealed test    : {len(X_test)} patients  "
      f"(PCOS=0: {(y_test==0).sum()}   PCOS=1: {(y_test==1).sum()})")

# ── Feature subsets ──────────────────────────────────────────────
with open(PATHS['SELECTED_FEAT']) as f:
    SELECTED = json.load(f)
SUBSET_COLS = {}
for name in ['PSO', 'HHO', 'MI', 'ALL']:
    feats   = SELECTED[name]['features']
    missing = [ft for ft in feats if ft not in name_to_idx]
    if missing:
        raise ValueError(f"{name}: features not found: {missing}")
    SUBSET_COLS[name] = [name_to_idx[ft] for ft in feats]

# ── Regime-aligned enhanced (tuned) hyperparameters ──────────────
with open(PATHS['ENHANCED_HP']) as f:
    ENHANCED = json.load(f)

# ── Frozen Track-1 default hyperparameters (the sealed-test anchor) ─
DEFAULTS = {
    'HIDDEN_DIM'   : 64,   'DROPOUT'      : 0.3,
    'LR'           : 0.001,'WEIGHT_DECAY' : 1e-4,
    'K_NEIGHBOURS' : 10,   'N2V_DIM'      : 64,
    'OPTIMISER'    : 'adam','SCHEDULER'   : 'plateau',
}

def _tuned_rc(subset, col_idx):
    p = ENHANCED[subset]['params']
    return {'col_idx': col_idx,
            'HIDDEN_DIM': p['hidden_dim'], 'DROPOUT': p['dropout'],
            'LR': p['lr'], 'WEIGHT_DECAY': p['weight_decay'],
            'K_NEIGHBOURS': p['k_neighbours'], 'N2V_DIM': p['n2v_dim'],
            'OPTIMISER': p['optimiser'], 'SCHEDULER': p['scheduler']}

def _default_rc(col_idx):
    rc = dict(DEFAULTS); rc['col_idx'] = col_idx; return rc

RUN_CONFIGS = {}
for subset in ['PSO', 'HHO', 'MI', 'ALL']:
    RUN_CONFIGS[f'{subset}_DEFAULT'] = _default_rc(SUBSET_COLS[subset])
    RUN_CONFIGS[f'{subset}_TUNED']   = _tuned_rc(subset, SUBSET_COLS[subset])

# BASELINE: all 48 features under defaults — identity twin of ALL_DEFAULT
RUN_CONFIGS['BASELINE'] = _default_rc(list(range(len(feature_names))))

RUN_ORDER = ['PSO_DEFAULT', 'PSO_TUNED', 'HHO_DEFAULT', 'HHO_TUNED',
             'MI_DEFAULT',  'MI_TUNED',  'ALL_DEFAULT', 'ALL_TUNED', 'BASELINE']

print("\nNine configurations assembled:")
for name in RUN_ORDER:
    rc = RUN_CONFIGS[name]
    print(f"  {name:<12}: {len(rc['col_idx']):>2} feat | hidden={rc['HIDDEN_DIM']}  "
          f"k={rc['K_NEIGHBOURS']}  n2v={rc['N2V_DIM']}  opt={rc['OPTIMISER']}  sch={rc['SCHEDULER']}")


Loaded  : 541 rows x 49 columns
Features: 48
PCOS=0  : 364   PCOS=1  : 177

Training pool  : 432 patients  (PCOS=0: 291  PCOS=1: 141)
Sealed test    : 109 patients  (PCOS=0: 73   PCOS=1: 36)

Nine configurations assembled:
  PSO_DEFAULT : 22 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau
  PSO_TUNED   : 22 feat | hidden=32  k=12  n2v=64  opt=adamw  sch=cosine
  HHO_DEFAULT : 17 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau
  HHO_TUNED   : 17 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau
  MI_DEFAULT  : 20 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau
  MI_TUNED    : 20 feat | hidden=32  k=20  n2v=64  opt=adam  sch=cosine
  ALL_DEFAULT : 48 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau
  ALL_TUNED   : 48 feat | hidden=128  k=8  n2v=64  opt=adamw  sch=cosine
  BASELINE    : 48 feat | hidden=64  k=10  n2v=64  opt=adam  sch=plateau


In [5]:
# CELL 6 — SEALED-TEST EVALUATION FUNCTION

def evaluate_config(config_name, rc, device):
    print(f"\n{'='*62}")
    print(f"  CONFIG: {config_name}  |  {len(rc['col_idx'])} features")
    print(f"  {FIXED['FINAL_EPOCHS']} epochs  |  No early stopping  |  {device}")
    print(f"{'='*62}")

    set_seed(FIXED['SEED'])
    col = rc['col_idx']

    # ── Per-run cfg ────────────────────────────────────────────────
    cfg = dict(FIXED)
    cfg['K_NEIGHBOURS'] = rc['K_NEIGHBOURS']
    cfg['N2V_DIM']      = rc['N2V_DIM']

    # ── Subset features (FULL training pool + test) ────────────────
    X_tr = X_train[:, col]
    X_te = X_test[:,  col]
    n_clinical = X_tr.shape[1]

    # ── Scale ──────────────────────────────────────────────────────
    print("  [1] Scaling...", end=' ', flush=True)
    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    print("done")

    # ── Build kNN graph from FULL training pool ────────────────────
    print("  [2] Building training graph...", end=' ', flush=True)
    ei_tr, ew_tr = build_knn_graph(X_tr_sc, k=cfg['K_NEIGHBOURS'])
    n_real = len(X_tr_sc)
    print(f"nodes={n_real}  edges={ei_tr.size(1)}")

    # ── Node2Vec ───────────────────────────────────────────────────
    print("  [3] Training Node2Vec...", end=' ', flush=True)
    n2v_tr = train_node2vec(ei_tr, n_real, cfg)
    print(f"dim={cfg['N2V_DIM']}")

    X_tr_full = np.concatenate([X_tr_sc, n2v_tr], axis=1)

    # ── SMOTE ──────────────────────────────────────────────────────
    print("  [4] Applying SMOTE...")
    X_tr_sm, y_tr_sm = apply_smote(X_tr_full, y_train, cfg['SEED'])
    n_syn       = len(X_tr_sm) - n_real
    n_train_aug = len(X_tr_sm)

    # ── Add synthetic nodes to graph ───────────────────────────────
    if n_syn > 0:
        X_syn_clin = X_tr_sm[n_real:, :n_clinical]
        ei_aug, ew_aug = add_synthetic_nodes(
            ei_tr, ew_tr, X_tr_sc, X_syn_clin, k=cfg['K_NEIGHBOURS'])
    else:
        ei_aug, ew_aug = ei_tr, ew_tr

    # ── Inductive N2V for test nodes ───────────────────────────────
    print("  [5] Inductive Node2Vec for test patients...", end=' ', flush=True)
    n2v_te    = inductive_n2v(X_te_sc, X_tr_sc, n2v_tr, k=cfg['K_NEIGHBOURS'])
    X_te_full = np.concatenate([X_te_sc, n2v_te], axis=1)
    print("done")

    # ── Add test nodes to graph BEFORE training ────────────────────
    ei_final, ew_final = add_test_nodes(
        ei_aug, ew_aug, X_tr_sc, X_te_sc,
        k=cfg['K_NEIGHBOURS'], n_train_aug=n_train_aug)

    # ── Build PyG Data (train+syn+test, all in graph) ──────────────
    X_all = np.concatenate([X_tr_sm, X_te_full], axis=0)
    y_all = np.concatenate([y_tr_sm, y_test])
    n_all = len(X_all)

    train_mask = torch.zeros(n_all, dtype=torch.bool)
    train_mask[:n_train_aug] = True
    test_mask  = torch.zeros(n_all, dtype=torch.bool)
    test_mask[n_train_aug:]  = True

    data = Data(
        x           = torch.tensor(X_all,  dtype=torch.float),
        edge_index  = ei_final,
        edge_weight = ew_final,
        y           = torch.tensor(y_all,  dtype=torch.long),
        train_mask  = train_mask,
        test_mask   = test_mask,
    ).to(device)

    # ── Class weights ──────────────────────────────────────────────
    n_neg = float((y_tr_sm == 0).sum())
    n_pos = float((y_tr_sm == 1).sum())
    cw    = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=cw)

    in_channels = n_clinical + cfg['N2V_DIM']
    print(f"\n  [6] PyG Data ready.")
    print(f"      Nodes: {n_all}  (train+syn={n_train_aug}, test={len(X_te)})")
    print(f"      Edges: {ei_final.size(1)}")
    print(f"      Features/node: {in_channels}  ({n_clinical} clinical + {cfg['N2V_DIM']} N2V)")

    # ── Model + optimiser + scheduler ──────────────────────────────
    model = GraphSAGE(in_channels, rc['HIDDEN_DIM'], 2, rc['DROPOUT']).to(device)

    if rc['OPTIMISER'] == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(),
                                       lr=rc['LR'], weight_decay=rc['WEIGHT_DECAY'])
    else:
        optimizer = torch.optim.Adam(model.parameters(),
                                      lr=rc['LR'], weight_decay=rc['WEIGHT_DECAY'])

    if rc['SCHEDULER'] == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=cfg['FINAL_EPOCHS'], eta_min=1e-6)
        plateau = False
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode='min', factor=0.5, patience=20, min_lr=1e-6)
        plateau = True

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"      Parameters: {n_params:,}")

    # ── Training: FIXED epochs, NO early stopping ──────────────────
    losses = []
    for epoch in range(1, cfg['FINAL_EPOCHS'] + 1):
        loss = train_one_epoch(model, data, optimizer, criterion)
        if loss != loss:
            print(f"    NaN loss at epoch {epoch}. Stopping.")
            break
        losses.append(loss)
        if plateau:
            scheduler.step(loss)
        else:
            scheduler.step()
        if epoch % 30 == 0:
            print(f"    Epoch {epoch:3d}/{cfg['FINAL_EPOCHS']}  "
                  f"loss={loss:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    print(f"  Training done. Final loss: {losses[-1]:.4f}")

    # ── Evaluate on sealed test set ────────────────────────────────
    preds_te, probs_te, true_te = evaluate_model(model, data, data.test_mask)
    metrics, (tn, fp, fn, tp) = compute_metrics(true_te, preds_te, probs_te)

    print(f"\n  Sealed-test result: {config_name}")
    print(f"      MCC       {metrics['mcc']:.4f}")
    print(f"      AUROC     {metrics['auroc']:.4f}")
    print(f"      Accuracy  {metrics['accuracy']:.4f}")
    print(f"      F1        {metrics['f1']:.4f}")
    print(f"      Sens      {metrics['sensitivity']:.4f}  |  Spec  {metrics['specificity']:.4f}")
    print(f"      TP={tp}  FP={fp}  TN={tn}  FN={fn}")

    # Save model weights
    torch.save(model.state_dict(),
               os.path.join(OUTPUT_DIR, f'GRAPHSAGE_{config_name}_final_weights.pt'))
    print(f"  Weights saved -> GRAPHSAGE_{config_name}_final_weights.pt")

    del model, data, optimizer, scheduler, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'config'      : config_name,
        'n_feat'      : n_clinical,
        'metrics'     : metrics,
        'cm'          : {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'losses'      : losses,
        'y_true'      : true_te.tolist(),
        'probs'       : probs_te.tolist(),
    }

print("Sealed-test evaluation function loaded.")

Sealed-test evaluation function loaded.


In [6]:
# CELL 7 — RUN ALL NINE CONFIGURATIONS 

all_results = {}
for name in RUN_ORDER:
    result = evaluate_config(name, RUN_CONFIGS[name], DEVICE)
    all_results[name] = result

    # Save incrementally
    out_path = os.path.join(OUTPUT_DIR, 'sealed_test_results.json')
    serialisable = {}
    for k, v in all_results.items():
        serialisable[k] = {
            'config'  : v['config'],
            'n_feat'  : v['n_feat'],
            'metrics' : v['metrics'],
            'cm'      : v['cm'],
        }
    with open(out_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f"  Saved -> {out_path}")

print(f"\n{'='*62}")
print("  All nine configurations evaluated on the sealed test set.")
print(f"{'='*62}")


  CONFIG: PSO_DEFAULT  |  22 features
  150 epochs  |  No early stopping  |  cuda
  [1] Scaling... done
  [2] Building training graph... nodes=432  edges=5524
  [3] Training Node2Vec... dim=64
  [4] Applying SMOTE...
  SMOTE: +150 synthetic PCOS+ (432 -> 582 total)
  [5] Inductive Node2Vec for test patients... done

  [6] PyG Data ready.
      Nodes: 691  (train+syn=582, test=109)
      Edges: 10704
      Features/node: 86  (22 clinical + 64 N2V)
      Parameters: 19,458
    Epoch  30/150  loss=0.2604  lr=1.00e-03
    Epoch  60/150  loss=0.1933  lr=1.00e-03
    Epoch  90/150  loss=0.1241  lr=1.00e-03
    Epoch 120/150  loss=0.0784  lr=1.00e-03
    Epoch 150/150  loss=0.0450  lr=1.00e-03
  Training done. Final loss: 0.0450

  Sealed-test result: PSO_DEFAULT
      MCC       0.6411
      AUROC     0.8988
      Accuracy  0.8440
      F1        0.7536
      Sens      0.7222  |  Spec  0.9041
      TP=26  FP=7  TN=66  FN=10
  Weights saved -> GRAPHSAGE_PSO_DEFAULT_final_weights.pt
  Saved ->

In [7]:
# CELL 8 — RESULTS TABLE (default vs tuned) + PER-SUBSET DELTA + IDENTITY CHECK
KEY_METRICS = ['mcc', 'auroc', 'accuracy', 'f1', 'sensitivity', 'specificity',
               'precision', 'recall', 'kappa', 'auprc', 'fmi', 'nmi']

print("\n" + "="*106)
print("  Sealed test — GraphSAGE: default vs tuned hyperparameters")
print("="*106)
header = f"  {'Config':<12} {'#Feat':>5}"
for m in KEY_METRICS:
    header += f"  {m.upper():>8}"
print(header)
print("-"*106)
for name in RUN_ORDER:
    r = all_results[name]
    row = f"  {name:<12} {r['n_feat']:>5}"
    for m in KEY_METRICS:
        row += f"  {r['metrics'][m]:>8.4f}"
    print(row)
print("="*106)

# ── Per-subset tuned - default delta (sealed-test MCC) ────────────
print("\n  Tuned - Default delta (sealed-test MCC):")
for subset in ['PSO', 'HHO', 'MI', 'ALL']:
    d = all_results[f'{subset}_DEFAULT']['metrics']['mcc']
    t = all_results[f'{subset}_TUNED']['metrics']['mcc']
    print(f"    {subset:<4}: default={d:.4f}  tuned={t:.4f}  delta={t - d:+.4f}")

# ── Protocol identity check: ALL_DEFAULT must equal BASELINE ──────
a = all_results['ALL_DEFAULT']['metrics']['mcc']
b = all_results['BASELINE']['metrics']['mcc']
diff = abs(a - b)
print(f"\n  IDENTITY CHECK  ALL_DEFAULT vs BASELINE  |MCC diff| = {diff:.2e}  "
      f"-> {'PASS' if diff < 1e-9 else 'FAIL — investigate before reporting'}")

# ── Headline = best default-regime config (the anchor) ───────────
best_default = max(['PSO_DEFAULT', 'HHO_DEFAULT', 'MI_DEFAULT', 'ALL_DEFAULT'],
                   key=lambda n: all_results[n]['metrics']['mcc'])
print(f"\n  Best DEFAULT-regime config by MCC: {best_default} "
      f"(MCC={all_results[best_default]['metrics']['mcc']:.4f})")
print("  Note: defaults are the reporting anchor; the tuned column tests whether")
print("        tuning improves on them.")



  Sealed test — GraphSAGE: default vs tuned hyperparameters
  Config       #Feat       MCC     AUROC  ACCURACY        F1  SENSITIVITY  SPECIFICITY  PRECISION    RECALL     KAPPA     AUPRC       FMI       NMI
----------------------------------------------------------------------------------------------------------
  PSO_DEFAULT     22    0.6411    0.8988    0.8440    0.7536    0.7222    0.9041    0.7879    0.7222    0.6398    0.8700    0.7644    0.3309
  PSO_TUNED       22    0.7143    0.9197    0.8716    0.8108    0.8333    0.8904    0.7895    0.8333    0.7137    0.9017    0.7937    0.4158
  HHO_DEFAULT     17    0.8148    0.9410    0.9174    0.8767    0.8889    0.9315    0.8649    0.8889    0.8147    0.9325    0.8611    0.5619
  HHO_TUNED       17    0.7736    0.9368    0.8991    0.8493    0.8611    0.9178    0.8378    0.8611    0.7735    0.9115    0.8337    0.4977
  MI_DEFAULT      20    0.8341    0.9456    0.9266    0.8889    0.8889    0.9452    0.8889    0.8889    0.8341    0.9252

In [8]:
# CELL 9 — CONFUSION MATRICES + LOSS CURVES
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ── Confusion matrices ───────────────────────────────────────────
fig, axes = plt.subplots(1, len(RUN_ORDER), figsize=(4 * len(RUN_ORDER), 4))
for ax, name in zip(axes, RUN_ORDER):
    cm_vals = all_results[name]['cm']
    cm = np.array([[cm_vals['tn'], cm_vals['fp']],
                   [cm_vals['fn'], cm_vals['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    mcc_val = all_results[name]['metrics']['mcc']
    ax.set_title(f"{name}\nMCC={mcc_val:.4f}", fontweight='bold')

plt.suptitle("Sealed Test — Confusion Matrices", fontweight='bold', y=1.02)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'sealed_test_confusion_matrices.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {cm_path}")

# ── Loss curves ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for name in RUN_ORDER:
    ax.plot(all_results[name]['losses'], label=name, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Curves — All Configs', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
loss_path = os.path.join(OUTPUT_DIR, 'sealed_test_loss_curves.png')
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {loss_path}")

Saved -> /kaggle/working/sealed_test_confusion_matrices.png
Saved -> /kaggle/working/sealed_test_loss_curves.png


In [9]:
# CELL 10 — SAVE RESULTS CSV + VERIFY

rows = []
for name in RUN_ORDER:
    r = all_results[name]
    row = {'config': name, 'n_features': r['n_feat']}
    row.update(r['metrics'])
    row.update(r['cm'])
    rows.append(row)

results_df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, 'sealed_test_results.csv')
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved -> {csv_path}")

# ── Verify JSON ──────────────────────────────────────────────────
json_path = os.path.join(OUTPUT_DIR, 'sealed_test_results.json')
with open(json_path) as f:
    loaded = json.load(f)
print(f"\nVerification — {json_path}:")
for name, r in loaded.items():
    print(f"  {name:<10}  MCC={r['metrics']['mcc']:.4f}  "
          f"AUROC={r['metrics']['auroc']:.4f}  "
          f"Acc={r['metrics']['accuracy']:.4f}  "
          f"F1={r['metrics']['f1']:.4f}")

print(f"\nAll outputs saved to {OUTPUT_DIR}")
print("Sealed-test results for the default-vs-tuned comparison saved.")

Results CSV saved -> /kaggle/working/sealed_test_results.csv

Verification — /kaggle/working/sealed_test_results.json:
  PSO_DEFAULT  MCC=0.6411  AUROC=0.8988  Acc=0.8440  F1=0.7536
  PSO_TUNED   MCC=0.7143  AUROC=0.9197  Acc=0.8716  F1=0.8108
  HHO_DEFAULT  MCC=0.8148  AUROC=0.9410  Acc=0.9174  F1=0.8767
  HHO_TUNED   MCC=0.7736  AUROC=0.9368  Acc=0.8991  F1=0.8493
  MI_DEFAULT  MCC=0.8341  AUROC=0.9456  Acc=0.9266  F1=0.8889
  MI_TUNED    MCC=0.7962  AUROC=0.9437  Acc=0.9083  F1=0.8649
  ALL_DEFAULT  MCC=0.8110  AUROC=0.9486  Acc=0.9174  F1=0.8696
  ALL_TUNED   MCC=0.7324  AUROC=0.9353  Acc=0.8807  F1=0.8219
  BASELINE    MCC=0.8110  AUROC=0.9486  Acc=0.9174  F1=0.8696

All outputs saved to /kaggle/working/
Sealed-test results for the default-vs-tuned comparison saved.


In [10]:
# CELL 11 — PER-SUBJECT ROC / PR / DET CURVES (default vs tuned) + RAW ARRAYS

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, precision_recall_curve, det_curve,
                             roc_auc_score, average_precision_score)

SUBSETS = ['PSO', 'HHO', 'MI', 'ALL']
COLOURS = {'DEFAULT': '#3a86ff', 'TUNED': '#ff595e'}
curve_arrays = {}

for subset in SUBSETS:
    fig, (ax_roc, ax_pr, ax_det) = plt.subplots(1, 3, figsize=(15, 4.5))
    curve_arrays[subset] = {}

    for regime in ['DEFAULT', 'TUNED']:
        r   = all_results[f'{subset}_{regime}']
        y   = np.asarray(r['y_true'])
        p   = np.asarray(r['probs'])
        col = COLOURS[regime]

        fpr, tpr, _     = roc_curve(y, p)
        prec, rec, _    = precision_recall_curve(y, p)
        fpr_d, fnr_d, _ = det_curve(y, p)
        auroc = roc_auc_score(y, p)
        auprc = average_precision_score(y, p)

        ax_roc.plot(fpr, tpr, color=col, lw=2, label=f"{regime.title()} (AUROC={auroc:.3f})")
        ax_pr.plot(rec, prec, color=col, lw=2, label=f"{regime.title()} (AUPRC={auprc:.3f})")
        ax_det.plot(fpr_d, fnr_d, color=col, lw=2, label=f"{regime.title()}")

        curve_arrays[subset][regime] = {
            'roc': {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auroc': float(auroc)},
            'pr' : {'recall': rec.tolist(), 'precision': prec.tolist(), 'auprc': float(auprc)},
            'det': {'fpr': fpr_d.tolist(), 'fnr': fnr_d.tolist()},
        }

    ax_roc.plot([0, 1], [0, 1], '--', color='grey', lw=1)
    ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title(f'{subset} - ROC'); ax_roc.legend(loc='lower right', fontsize=9)
    ax_roc.grid(True, alpha=0.3)

    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_title(f'{subset} - Precision-Recall'); ax_pr.legend(loc='lower left', fontsize=9)
    ax_pr.grid(True, alpha=0.3)

    ax_det.set_xlabel('False Positive Rate'); ax_det.set_ylabel('False Negative Rate')
    ax_det.set_title(f'{subset} - DET'); ax_det.legend(loc='upper right', fontsize=9)
    ax_det.grid(True, alpha=0.3)

    fig.suptitle(f'{subset}: default vs regime-aligned tuned (sealed test)', fontweight='bold', y=1.02)
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, f'sealed_test_curves_{subset}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"Saved -> {fig_path}")

curve_path = os.path.join(OUTPUT_DIR, 'sealed_test_curve_arrays.json')
with open(curve_path, 'w') as f:
    json.dump(curve_arrays, f, indent=2)
print(f"Raw curve arrays saved -> {curve_path}")


Saved -> /kaggle/working/sealed_test_curves_PSO.png
Saved -> /kaggle/working/sealed_test_curves_HHO.png
Saved -> /kaggle/working/sealed_test_curves_MI.png
Saved -> /kaggle/working/sealed_test_curves_ALL.png
Raw curve arrays saved -> /kaggle/working/sealed_test_curve_arrays.json


In [11]:
# CELL 12 — PERSIST PER-PATIENT VECTORS (paired-test input)
per_patient = {
    cfg: {'y_true': res['y_true'], 'probs': res['probs']}
    for cfg, res in all_results.items()
}
pp_path = os.path.join(OUTPUT_DIR, 'per_patient_sealed_predictions_FS.json')
with open(pp_path, 'w') as f:
    json.dump(per_patient, f)
print(f"Per-patient vectors saved -> {pp_path}  ({len(per_patient)} configs)")

Per-patient vectors saved -> /kaggle/working/per_patient_sealed_predictions_FS.json  (9 configs)


In [12]:
# CELL 13 — ONE-CLICK ZIP OF ALL OUTPUTS
import zipfile, os

output_dir  = '/kaggle/working/'
zip_path    = '/kaggle/working/PCOS_FS_Eval_Results.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir(output_dir):
        if not file.endswith('.zip'):
            file_path = os.path.join(output_dir, file)
            if os.path.isfile(file_path):
                zipf.write(file_path, file)
                print(f"  Added: {file}")

print(f"\nZip created: {zip_path}")
print(f"Total files: {len([f for f in os.listdir(output_dir) if not f.endswith('.zip')])}")

zip_path

  Added: GRAPHSAGE_BASELINE_final_weights.pt
  Added: sealed_test_results.json
  Added: sealed_test_loss_curves.png
  Added: GRAPHSAGE_ALL_TUNED_final_weights.pt
  Added: GRAPHSAGE_HHO_TUNED_final_weights.pt
  Added: GRAPHSAGE_PSO_TUNED_final_weights.pt
  Added: sealed_test_curves_ALL.png
  Added: GRAPHSAGE_PSO_DEFAULT_final_weights.pt
  Added: sealed_test_curves_PSO.png
  Added: per_patient_sealed_predictions_FS.json
  Added: GRAPHSAGE_MI_DEFAULT_final_weights.pt
  Added: GRAPHSAGE_HHO_DEFAULT_final_weights.pt
  Added: sealed_test_confusion_matrices.png
  Added: sealed_test_curve_arrays.json
  Added: sealed_test_curves_MI.png
  Added: GRAPHSAGE_ALL_DEFAULT_final_weights.pt
  Added: GRAPHSAGE_MI_TUNED_final_weights.pt
  Added: sealed_test_results.csv
  Added: sealed_test_curves_HHO.png

Zip created: /kaggle/working/PCOS_FS_Eval_Results.zip
Total files: 20


'/kaggle/working/PCOS_FS_Eval_Results.zip'